**Download and align reads to the mouse transcriptome using kallisto**

First download the mouse transcriptome data from Ensemble.

This file contains transcript sequences only (expressed RNA transcripts converted to cDNA).

In [ ]:
%%bash

OUTDIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna"

mkdir -p "$OUTDIR"

wget -P "$OUTDIR" \
ftp://ftp.ensembl.org/pub/release-97/fasta/mus_musculus/cdna/Mus_musculus.GRCm38.cdna.all.fa.gz

**Create a new folder named kallisto:**

In [ ]:
%%bash

KALLISTO_DIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations"

mkdir -p "$KALLISTO_DIR"

**Run the indexing command. This prepares the transcriptome so that we can peudoalign reads to it. This will take a few minutes.:**

In [ ]:
%%bash

REF_DIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna"

kallisto index --index="$REF_DIR/Mus_musculus.GRCm38_index" "$REF_DIR"/Mus_musculus.GRCm38.cdna.all.fa.gz

**Move the kallisto index file to the transcriptome folder:**

In [ ]:
%%bash

IN_DIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna"

OUT_DIR="/var/www/html/usb/jupyter/my_projects/transcriptome"

mkdir -p "$OUT_DIR"

mv "$IN_DIR"/Mus_musculus.GRCm38_index "$OUT_DIR"

**Download the mouse GTF file, a file containing coordinates and descriptions for all gene names and locations:**

In [ ]:
%%bash

OUTDIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna"

mkdir -p "$OUTDIR"

wget -P "$OUTDIR" ftp://ftp.ensembl.org/pub/release-97/gtf/mus_musculus/Mus_musculus.GRCm38.97.chr.gtf.gz

**Move the GTF file to the kallisto/annotations folder:**

In [ ]:
%%bash

REFDIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna"
OUTDIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations"

mv "$REFDIR"/Mus_musculus.GRCm38.97.chr.gtf.gz "$OUTDIR"

**Download a textfile that has the name of each mouse chromosome (e.g. 1, 2, 3, ... X, Y, MT) and the length of each chromosome:**

In [ ]:
%%bash

OUTDIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations" 

mkdir -p "$OUTDIR"

wget -P "$OUTDIR" ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/635/GCF_000001635.26_GRCm38.p6/GCF_000001635.26_GRCm38.p6_assembly_report.txt

**Extract the first 64 lines, then only keep the last 21 lines of those 64:**

head -64n = Extract the first 64 lines

tail -n21 = Extract the last 21 lines

In [ ]:
%%bash

REFDIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations" 

head -n63 "$REFDIR"/GCF_000001635.26_GRCm38.p6_assembly_report.txt|tail -n21|cut -f1,9 > "$REFDIR"/mouse_chromosomes.tsv

**Display the extracted list of chromosome numbers in column 1 and chromosome lengths in column 2:**

In [ ]:
%%bash

REFDIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations" 

cat "$REFDIR"/mouse_chromosomes.tsv

**Pseudoaligning reads with Kallisto:**

Working with pre-trimmed data:

All instructions for the commands we are using are in the Kallisto manual: https://pachterlab.github.io/kallisto/manual. Since we are using single read data, we need to provide information on the fragment length used for the library (200) and an estimate of the standard deviation for this value - here we will have to guess (20). We need to run Kallisto separately on each of our 6 files so we will use a for loop. This will take up to 15 minutes per sample. 

In [ ]:
%%bash

TRIMMED_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/fastq-trimmed"

TRANSCRIPTOME_DIR="/var/www/html/usb/jupyter/my_projects/reference_genomes/mus_musculus/rna_cdna/Mus_musculus.GRCm38_index"

GTF_DIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations"

CHR_DIR="/var/www/html/usb/jupyter/my_projects/kallisto/annotations"

OUTDIR="/var/www/html/usb/jupyter/my_projects/kallisto/analyzed"

mkdir -p "$OUTDIR"

for file in "$TRIMMED_DIR"/*.fastq.gz
do
    sample=$(basename "$file" .fastq.gz)

    kallisto quant \
        --single \
        --threads=4 \
        --index="$TRANSCRIPTOME_DIR" \
        --bootstrap-samples=25 \
        --fragment-length=200 \
        --sd=20 \
        --output-dir="$OUTDIR/${sample}_quant" \
        --genomebam \
        --gtf="$GTF_DIR/Mus_musculus.GRCm38.97.chr.gtf.gz" \
        --chromosomes="$CHR_DIR/mouse_chromosomes.tsv" \
        "$file"

done

**View the list of abundances (counts) for the sample:**

This is just for testing purposes...

In [ ]:
%%bash

REFDIR="/var/www/html/usb/jupyter/my_projects/kallisto/analyzed/SRR5017128_quant"

head -n 100 "$REFDIR"/abundance.tsv